# Graph Traversal in Python





## 1. What is a Graph?

A **graph** is a data structure consisting of:

- **Vertices (nodes)** – entities or objects.
- **Edges** – relationships/connections between nodes.

For example:

```text
A ----- B
|       |
|       |
C ----- D
```

Here:

- `A`, `B`, `C`, and `D` are nodes.
- The lines between them are edges.

Graphs are useful for representing relationships such as:

- Social networks
- Road networks
- Computer networks
- Web page links
- ETL pipeline dependencies
- Data lineage
- Workflow dependencies


## 2. Types of Graphs

### Undirected graph

The relationship works in both directions.

```text
A ----- B
```

If `A` is connected to `B`, then `B` is also connected to `A`.

### Directed graph

The relationship has a direction.

```text
A -----> B
```

This is especially useful for **data pipelines** because a transformation may depend on another task:

```text
Extract → Transform → Load
```

### Weighted graph

Edges have values such as distance, cost, or time.

```text
A --5--> B
```

For this demo, we will mainly work with **unweighted graphs**, especially directed graphs for Data Engineering examples.


## 3. Representing a Graph in Python

One of the most useful representations in Python is an **adjacency list**.

We can use a dictionary:

```python
graph = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["E"],
    "D": [],
    "E": []
}
```

The key represents a node and the list contains its neighboring nodes.


In [1]:
graph = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["E"],
    "D": [],
    "E": []
}

print(graph)


{'A': ['B', 'C'], 'B': ['D'], 'C': ['E'], 'D': [], 'E': []}


### Visual representation

The graph above can be visualized as:

```text
        A
       / \
      B   C
      |   |
      D   E
```

The adjacency list is convenient because we can quickly access the neighbors of a node.


## 4. Adding Nodes and Edges

Let's create a small helper function for an undirected graph.

This demonstrates how Python dictionaries and lists can be used to dynamically construct a graph.


In [2]:
graph = {}

def add_node(graph, node):
    if node not in graph:
        graph[node] = []

def add_edge(graph, u, v):
    add_node(graph, u)
    add_node(graph, v)
    graph[u].append(v)
    graph[v].append(u)

add_edge(graph, "A", "B")
add_edge(graph, "A", "C")
add_edge(graph, "B", "D")
add_edge(graph, "C", "E")

print(graph)


{'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A', 'E'], 'D': ['B'], 'E': ['C']}


## 5. Breadth-First Search (BFS)

**BFS** explores a graph level by level.

For the graph:

```text
        A
       / \
      B   C
      |   |
      D   E
```

Starting from `A`:

```text
Level 0: A
Level 1: B, C
Level 2: D, E
```

So the traversal is:

```text
A → B → C → D → E
```

### Key idea

BFS uses a **queue**.

A queue follows:

> First In, First Out (FIFO)

Python's `collections.deque` is a good choice for implementing a queue efficiently.


In [3]:
from collections import deque

def bfs(graph, start):
    visited = set()
    queue = deque([start])
    visited.add(start)

    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order

graph = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["E"],
    "D": [],
    "E": []
}

print("BFS:", bfs(graph, "A"))


BFS: ['A', 'B', 'C', 'D', 'E']


### Why do we need `visited`?

Graphs can contain cycles:

```text
A → B → C
    ↑   |
    └───┘
```

Without a `visited` set, the algorithm could repeatedly visit the same nodes.

The `visited` set ensures that every node is processed at most once.


## 6. Depth-First Search (DFS)

**DFS** explores one path as deeply as possible before backtracking.

For example:

```text
        A
       / \
      B   C
      |   |
      D   E
```

A possible DFS traversal is:

```text
A → B → D → C → E
```

DFS can be implemented using:

1. Recursion
2. An explicit stack

We will first use recursion.


In [4]:
def dfs(graph, node, visited=None):
    if visited is None:
        visited = set()

    visited.add(node)
    order = [node]

    for neighbor in graph[node]:
        if neighbor not in visited:
            order.extend(dfs(graph, neighbor, visited))

    return order

print("DFS:", dfs(graph, "A"))


DFS: ['A', 'B', 'D', 'C', 'E']


## 7. BFS vs DFS

| Feature | BFS | DFS |
|---|---|---|
| Main data structure | Queue | Stack / Recursion |
| Traversal | Level by level | Depth first |
| Shortest path in unweighted graph | Yes | Not necessarily |
| Useful for | Minimum-hop problems | Dependency/deep exploration |
| Time complexity | `O(V + E)` | `O(V + E)` |
| Space complexity | `O(V)` | `O(V)` |

Both algorithms are fundamental graph traversal techniques.


## 8. Finding a Path Using BFS

A common graph problem is:

> Can we reach node `X` from node `Y`?

We can extend BFS to store the path.

Example:

```text
A → B → D
```

We want to find the path from `A` to `D`.


In [5]:
from collections import deque

def bfs_path(graph, start, target):
    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        node, path = queue.popleft()

        if node == target:
            return path

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))

    return None

graph = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["E"],
    "D": [],
    "E": []
}

print("Path:", bfs_path(graph, "A", "D"))


Path: ['A', 'B', 'D']


# Data Engineering Application

## 9. Graphs in Data Engineering

Graphs are not only theoretical DSA structures.

They naturally appear in Data Engineering systems.

Examples include:

- ETL/ELT task dependencies
- Data lineage
- Workflow orchestration
- Table dependencies
- Pipeline impact analysis
- Job scheduling
- Dependency management

Consider this pipeline:

```text
Raw Data
    ↓
Cleaning
    ↓
Transformation
    ↓
Validation
    ↓
Data Warehouse
    ↓
Dashboard
```

Each processing step can be represented as a **node**, and a dependency can be represented as a **directed edge**.


In [6]:
pipeline = {
    "raw_data": ["cleaning"],
    "cleaning": ["transformation"],
    "transformation": ["validation", "feature_engineering"],
    "validation": ["warehouse"],
    "feature_engineering": ["warehouse"],
    "warehouse": ["dashboard"],
    "dashboard": []
}

print("BFS:", bfs(pipeline, "raw_data"))
print("DFS:", dfs(pipeline, "raw_data"))


BFS: ['raw_data', 'cleaning', 'transformation', 'validation', 'feature_engineering', 'warehouse', 'dashboard']
DFS: ['raw_data', 'cleaning', 'transformation', 'validation', 'warehouse', 'dashboard', 'feature_engineering']


## 10. Downstream Dependency Analysis

Suppose the `transformation` task changes.

A Data Engineer may want to know:

> Which downstream tasks could be affected?

We can use graph traversal to find all reachable downstream nodes.


In [7]:
def downstream_tasks(graph, task):
    return bfs(graph, task)[1:]

affected = downstream_tasks(pipeline, "transformation")

print("Tasks affected by transformation:")
for task in affected:
    print("-", task)


Tasks affected by transformation:
- validation
- feature_engineering
- warehouse
- dashboard


This is a simple example of **impact analysis**.

In a real data platform, the same idea can be applied to:

- Tables
- Views
- ETL jobs
- Data quality checks
- Reports
- Machine learning features

If one upstream dataset changes, graph traversal can help identify the downstream assets that may need to be checked or recomputed.


## 11. Data Lineage

**Data lineage** describes how data moves from its source to its final destination.

For example:

```text
customers.csv
      ↓
clean_customers
      ↓
customer_table
      ↓
customer_features
      ↓
ML_model
      ↓
predictions
```

This can be represented as a directed graph.

Graph traversal can answer questions such as:

- Where did this dataset come from?
- Which outputs depend on this table?
- What will be affected if a source changes?
- Is a target dataset reachable from a particular source?


In [8]:
lineage = {
    "customers.csv": ["clean_customers"],
    "clean_customers": ["customer_table"],
    "customer_table": ["customer_features"],
    "customer_features": ["ML_model"],
    "ML_model": ["predictions"],
    "predictions": []
}

print("Data lineage:")
print(" → ".join(bfs(lineage, "customers.csv")))


Data lineage:
customers.csv → clean_customers → customer_table → customer_features → ML_model → predictions


## 12. Reading Graph Data from a CSV

In real projects, graph relationships may come from files or databases instead of being hard-coded.

For example, an edge file might contain:

```text
source,target
raw_customers,clean_customers
raw_orders,clean_orders
clean_customers,customer_transform
clean_orders,order_transform
customer_transform,warehouse
order_transform,warehouse
warehouse,dashboard
```

We can use **Pandas** to read this data and construct the graph.

This connects:

```text
CSV → Pandas → Graph → BFS/DFS → Dependency Analysis
```


In [9]:
import pandas as pd
from io import StringIO

csv_data = StringIO("""source,target
raw_customers,clean_customers
raw_orders,clean_orders
clean_customers,customer_transform
clean_orders,order_transform
customer_transform,warehouse
order_transform,warehouse
warehouse,dashboard
""")

df = pd.read_csv(csv_data)
df


,source,target
0,raw_customers,clean_customers
1,raw_orders,clean_orders
2,clean_customers,customer_transform
3,clean_orders,order_transform
4,customer_transform,warehouse
5,order_transform,warehouse
6,warehouse,dashboard


In [10]:
def build_directed_graph(df):
    graph = {}

    for _, row in df.iterrows():
        source = row["source"]
        target = row["target"]

        graph.setdefault(source, [])
        graph.setdefault(target, [])

        graph[source].append(target)

    return graph

data_graph = build_directed_graph(df)

print(data_graph)


{'raw_customers': ['clean_customers'], 'clean_customers': ['customer_transform'], 'raw_orders': ['clean_orders'], 'clean_orders': ['order_transform'], 'customer_transform': ['warehouse'], 'order_transform': ['warehouse'], 'warehouse': ['dashboard'], 'dashboard': []}


In [11]:
print("BFS from raw_customers:")
print(bfs(data_graph, "raw_customers"))

print("\nDFS from raw_customers:")
print(dfs(data_graph, "raw_customers"))


BFS from raw_customers:
['raw_customers', 'clean_customers', 'customer_transform', 'warehouse', 'dashboard']

DFS from raw_customers:
['raw_customers', 'clean_customers', 'customer_transform', 'warehouse', 'dashboard']


# 13. Cycle Detection

Data pipelines are often represented as **DAGs**:

> Directed Acyclic Graphs

A DAG must not contain a cycle.

Valid:

```text
Extract → Transform → Validate → Load
```

Invalid:

```text
Extract → Transform → Validate
              ↑             |
              └─────────────┘
```

A cycle can create problems because a task may depend on itself indirectly.

For example:

```text
A → B → C → A
```

A DFS-based cycle detection algorithm can detect this situation.


In [12]:
def has_cycle(graph):
    visited = set()
    recursion_stack = set()

    def visit(node):
        visited.add(node)
        recursion_stack.add(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                if visit(neighbor):
                    return True
            elif neighbor in recursion_stack:
                return True

        recursion_stack.remove(node)
        return False

    for node in graph:
        if node not in visited:
            if visit(node):
                return True

    return False

valid_pipeline = {
    "extract": ["transform"],
    "transform": ["validate"],
    "validate": ["load"],
    "load": []
}

cyclic_pipeline = {
    "extract": ["transform"],
    "transform": ["validate"],
    "validate": ["transform"]
}

print("Valid pipeline has cycle:", has_cycle(valid_pipeline))
print("Cyclic pipeline has cycle:", has_cycle(cyclic_pipeline))


Valid pipeline has cycle: False
Cyclic pipeline has cycle: True


# 14. Mini Data Engineering Exercise

Consider this pipeline:

```text
raw_sales
    ↓
clean_sales
    ↓
sales_transform
   /       \
quality   aggregate
   \       /
    warehouse
       ↓
    dashboard
```

### Tasks

1. Build the graph using a Python dictionary.
2. Perform BFS starting from `raw_sales`.
3. Perform DFS starting from `raw_sales`.
4. Find all downstream tasks of `sales_transform`.
5. Check whether `dashboard` is reachable from `raw_sales`.
6. Check whether the pipeline contains a cycle.
7. Modify the graph to introduce a cycle and verify that cycle detection catches it.


In [13]:
exercise_pipeline = {
    "raw_sales": ["clean_sales"],
    "clean_sales": ["sales_transform"],
    "sales_transform": ["quality", "aggregate"],
    "quality": ["warehouse"],
    "aggregate": ["warehouse"],
    "warehouse": ["dashboard"],
    "dashboard": []
}

print("BFS:", bfs(exercise_pipeline, "raw_sales"))
print("DFS:", dfs(exercise_pipeline, "raw_sales"))

print("\nDownstream of sales_transform:")
print(downstream_tasks(exercise_pipeline, "sales_transform"))

print("\nDashboard reachable:",
      "dashboard" in bfs(exercise_pipeline, "raw_sales"))

print("Has cycle:", has_cycle(exercise_pipeline))


BFS: ['raw_sales', 'clean_sales', 'sales_transform', 'quality', 'aggregate', 'warehouse', 'dashboard']
DFS: ['raw_sales', 'clean_sales', 'sales_transform', 'quality', 'warehouse', 'dashboard', 'aggregate']

Downstream of sales_transform:
['quality', 'aggregate', 'warehouse', 'dashboard']

Dashboard reachable: True
Has cycle: False
